# 02 — RayDINO vs RAD-DINO, frozen encoders, matched head

**Do not assume RayDINO is better. It may improve, match or worsen performance.**

| | RAD-DINO | RayDINO (XRay-DINO) |
|---|---|---|
| arch | ViT-B/14 | ViT-L/16 |
| dim | 768 | 1024 (expected) |
| input | 518 px, 37x37 grid | read from the model card |
| weights | MIT, open | **FAIR Noncommercial Research License, gated** |

Patch grids and dimensions differ by construction, so **the primary comparison is
matched GLOBAL heads**. A regional comparison is secondary and uses the declared
`pool_to_grid` adaptive pooling to a common 12x12 grid.

Heads are fitted on TBX11K **train** only; regularisation is chosen on **development**
only. `internal_test` is never touched. Vietnam REPORT is never read.


In [ ]:
import os, sys, json, time, hashlib, platform
from pathlib import Path
WORK = Path('/kaggle/working'); INP = Path('/kaggle/input')
CODE = next((p for p in INP.glob('*/src/guards.py')), None)
if CODE is None:
    raise SystemExit('attach the code dataset; see KAGGLE_INPUT_CHECKLIST.md')
sys.path.insert(0, str(CODE.parent))
import guards as G
print('code from', CODE.parent)


In [ ]:
# --- target label: refuse to train on an undefined target
import yaml, json
from pathlib import Path
cfg = next((p for p in INP.glob('*/configs/target_label.yaml')), None)
if cfg is None:
    raise SystemExit('target_label.yaml missing - refusing to train')
TARGET = yaml.safe_load(Path(cfg).read_text())
G.assert_target_mapping(TARGET)
print(json.dumps(TARGET, indent=2))
print('BOTH arms use this identical mapping. It is not fixed in one arm only.')


In [ ]:
# --- splits, with the locked partition dropped before use
import pandas as pd
c01 = next(INP.rglob('provisional_c01.csv'))
df = pd.read_csv(c01)
usable, n_locked = G.drop_locked_rows(df)
print('locked rows excluded (count only):', n_locked)
train = usable[usable.split == 'train']
dev = usable[usable.split == 'development']
pos, neg = set(TARGET['positive_categories']), set(TARGET['negative_categories'])
train = train[train.category.isin(pos | neg)]
dev = dev[dev.category.isin(pos | neg)]
ytr = train.category.isin(pos).astype(int).values
ydv = dev.category.isin(pos).astype(int).values
print('train %d (%d pos) | development %d (%d pos)'
      % (len(ytr), ytr.sum(), len(ydv), ydv.sum()))


In [ ]:
# --- feature extraction, resumable, per encoder, with its OWN preprocessing
import encoders as EN, numpy as np, torch
WORK.mkdir(exist_ok=True, parents=True)
RAYDINO_WEIGHTS = os.environ.get('RAYDINO_WEIGHTS', '')  # user-supplied
def build(which):
    if which == 'raddino':
        m, proc, meta = EN.load_rad_dino()
        def enc(paths):
            from PIL import Image
            ims = [Image.open(p).convert('RGB') for p in paths]
            x = proc(images=ims, return_tensors='pt')['pixel_values'].cuda()
            with torch.no_grad():
                return m(pixel_values=x).pooler_output.float().cpu().numpy()
        return enc, meta
    m, _, meta = EN.load_raydino(RAYDINO_WEIGHTS)
    def enc(paths):
        from PIL import Image
        import torchvision.transforms as T
        px = meta['input_px']
        if not isinstance(px, int):
            raise SystemExit('set the RayDINO input resolution from its model card')
        tf = T.Compose([T.Resize((px, px)), T.ToTensor(),
                        T.Normalize([0.485]*3, [0.229]*3)])
        x = torch.stack([tf(Image.open(p).convert('RGB')) for p in paths]).cuda()
        with torch.no_grad():
            return m(x).float().cpu().numpy()
    return enc, meta
print('Set RAYDINO_WEIGHTS to the gated checkpoint path before running the',
      'RayDINO arm. Without it the arm is skipped, not faked.')


In [ ]:
# --- fit the matched head and score, per encoder
import evaluate as E
results = {}
for which in ('raddino', 'raydino'):
    try:
        enc, meta = build(which)
    except Exception as e:
        print('%s arm UNAVAILABLE: %s' % (which, e)); continue
    # ... extract features for train/dev/TUNE via EN.extract_features(...)
    # ... G.assert_ids_aligned(feature_ids, split_ids)  <- never positional
    # ... sel = E.select_C(Xtr, ytr, Xdv, ydv)
    # ... clf = E.fit_head(Xtr, ytr, c=sel['C']);  store clf with joblib
    # ... thr = E.fit_threshold(dev_scores, ydv, 'development')
    print('%s arm: wire the extraction loop, then evaluate' % which)
print()
print('Primary endpoint and decision rule: see DECISION_RULES.md.')
print('A small AUROC gain is NOT a win without the prespecified minimum effect,')
print('a paired interval excluding zero, and no unacceptable sensitivity loss in')
print('the previous-TB subgroup.')
